In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
    summarize_index_overlap,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`DSO` and {term}`IQTIG` data is not connected (see [](general:ic)). 

In [ ]:
split = split_data(data, ["donor_et_dso", "donor_et_iqtig"])
assert len(split) == 2, "Not 2 different row types present!?"

However, even though both sources provide longitudinal data, {term}`IQTIG` provides no date information.

In [ ]:
iqtig = split["donor_et_iqtig"].reset_index()
display_data_doc(data=iqtig)

The following analysis shows the overlap between the data contributors. We keep both, as there are some donors who only occur in the {term}`IQTIG` dataset.

In [ ]:
summarize_index_overlap(split["donor_et_iqtig"], split["donor_et_dso"], "IQTIG", "DSO")

In [ ]:
del split

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of tests (see [](general:rf)).  However, only {term}`DSO` provides a date for the measurement. We kept all rows.

In [ ]:
data["Institute with a measurement date"] = ((~data["sampling_date"].isna())).replace(
    {1: "Yes", 0: "No"}
)
data["donor"] = donors[donors.isin(targetpop["donor_et_id_et"])]
display_long_data_doc(
    data,
    [
        "donor",
    ],
    "sampling_date",
    "Institute with a measurement date",
)
data.drop(
    columns=["donor", "Institute with a measurement date"],
    inplace=True,
)

### Unit Conversions

We applied the common translations (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

### Consolidating Columns

We consolidated columns that appear for {term}`DSO` and {term}`IQTIG` (see [](general:crc))

In [ ]:
red = find_redundant_cols(data)
red.pop("donor_et")
red["donor_et_id_et"] = ["donor_et_dso", "donor_et_iqtig"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `sampling_date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["sampling_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important
class DonorPostmortemLabBloodGroup(SpenderID):
    antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Anitbodies Tested",
        description="Were the antibodies used for testing?",
        isin=["no", "yes"],
    )
    bloodgroup: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Blood group",
        description="The donor blood group",
        isin=["A", "AB", "B", "0"],
    )
    communicated_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Communicated date",
        description="Date when the lab result was communicated",
    )
    examination_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination date",
        description="Date when the lab measurements were conducted",
    )
    result_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Result date",
        description="Date when the lab result was generated",
    )
    rhesus: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Rhesus factor",
        description="The donor blood Rhesus factor",
        isin=["positive", "negative"],
    )
    sample_tissue: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Tissue",
        description="From what tissue was the sample taken?",
        isin=["Blut", "Serum", "Biopsie", "Heparinblut", "Gewebe", "Drainage"],
    )
    sampling_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Sampling date",
        description="Date when the sample was taken",
    )
    sampling_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Lab Type Test",
        description="What kind of test was used?",
        isin=["Serologische Bestimmung", "Bedside-Test", "PCR"],
    )

    class Config:
        title = "Donor Postmortem Blood Group Dataset"
        description = "Each row represents a bloodgroup lab test for a deceased donor. The data is based on the 'element_spender_postmortem_labor_blutgruppe.csv' file. It contains data from the IQTIG and DSO."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemLabBloodGroup, data)

In [ ]:
DonorPostmortemLabBloodGroup.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemLabBloodGroup.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)